# CS 789 — Assignment 7: SOC Copilot II — Automation, an Uncensored Model, the Finished Product
**AI-Powered Cyber Attacks and Defenses · Fall 2026 · UNLV · Prof. Yoohwan Kim**

A7 continues the copilot you built in A6 (same data, same KG, your own retriever, prompt and memory). Three
independent parts, then the finished product:

| Part | What you build | What you learn |
|---|---|---|
| 0 | Bring back **your A6 copilot** (`my_a6.py`) | incremental product development |
| A | An **n8n** workflow that triages the live alert feed; you add the **LLM node** in the n8n editor | AI inside automation pipelines |
| B | An **uncensored copy** of the copilot's model with **Heretic** (abliteration), measured against the original | what abliteration removes, what it costs, and where guardrails must live |
| C | Your own **dashboards** and the **finished SOC Copilot**, with an end-to-end acceptance test | putting it all together |

Parts A and B do not depend on each other; do A first (it needs Qwen running, B takes the GPU for a while).

> Runtime: **Colab Pro, L4 GPU** (an A100 makes Part B about twice as fast). About 2½ hours of your time plus
> about 40 minutes of unattended runs (Heretic). Everything runs inside your Colab VM.

---
## Part 0 — Setup and your A6 copilot (≈ 15 min)
Run 0.1 (the same setup as A6, from the A7 repository). Then 0.2 brings back your A6 work: upload the
**`my_a6.py`** you saved in A6 E.4 into `/content/A7` (📁 → upload), or keep it in Drive at `MyDrive/cs789/my_a6.py`
(mount Drive with the 📁 → Drive button first). If your A6 copilot was incomplete, set `USE_REFERENCE = True` to use
the instructor's A6 — that costs no points in A7.

In [ ]:
# 0.1 Files, packages, the local model server, and the data
import os, sys, importlib.util
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/A7"):
        !git clone -q https://github.com/unlv-cs789-f26/A7.git /content/A7
    %cd /content/A7
    if importlib.util.find_spec("llama_index") is None:
        !pip -q install -r requirements.txt
sys.path.insert(0, os.getcwd())
import soc6 as H
import pandas as pd
pd.set_option("display.max_colwidth", 120)
H.install_vllm()
H.ensure_model("qwen")
print(H.chat("qwen", [{"role": "user", "content": "Which ATT&CK technique ID is 'Valid Accounts'? One line."}])["text"])

In [ ]:
# 0.2 Your A6 copilot: build_kg + kg, kg_context, SYSTEM_PROMPT, answer, and the LangChain memory (chat_turn)
import shutil
from pathlib import Path
USE_REFERENCE = False
drive_copy = Path("/content/drive/MyDrive/cs789/my_a6.py")
if not Path("my_a6.py").exists() and drive_copy.exists():
    shutil.copy(drive_copy, "my_a6.py")
src = Path("my_a6.py")
if USE_REFERENCE or not src.exists():
    print("Using the REFERENCE A6 copilot")
    src = H.HERE / "reference" / "a6_reference.py"
exec(src.read_text())
H.check_kg(kg)
print(chat_turn("a7-smoke", "Who owns the host vpn?")["answer"])

---
## Part A — Automation with n8n (≈ 40 min)

A SOC does not wait for an analyst to ask: alerts arrive continuously and must be **triaged**. You will run
**n8n** (the workflow tool from lecture 5-2) inside the Colab VM. The provided workflow receives each alert on a
webhook, calls **your** `triage()` through the copilot's API, branches on priority, and opens a **case** for P1/P2.
One node is a placeholder: you will replace it with an **LLM node** in the n8n editor, so every case gets an
AI-written summary.

**Triage rubric** (the check uses it): **P1** level ≥ 10 on a host with criticality ≥ 4 · **P2** level ≥ 10
(any host), or level ≥ 8 on criticality ≥ 4 · **P3** level ≥ 6 · **P4** otherwise.

In [ ]:
# A.0 Install and start n8n (Node.js 24 + n8n, ~4 min the first time), and start the copilot's API on port 8080
H.install_n8n()
H.register(kg=kg, chat_turn=chat_turn, conversations_db=DB, models=["qwen"])
H.start_platform()          # your A6 copilot's server: n8n calls its triage API; Part C adds your dashboards
H.start_n8n()

#### 🛠️ A.1 triage(alert)

**Contract** (the checks below depend on these exact names): `triage(alert, model='qwen') -> {'alert_id', 'host', 'owner', 'criticality', 'priority', 'reason'}`

**🤖 Prompt for your AI assistant** — copy it into Claude in Chrome, Gemini in Chrome or Gemini in Colab, adapt it, and have the assistant write the code into the empty cell below and run it. Keep the prompt you finally used: it goes into your report.

```text
In this Colab notebook kg is a LlamaIndex SimplePropertyGraphStore; X = H.index(kg) gives X.nodes (dict
name -> (label, properties)) and X.neighbors(name, rel, direction) -> list of (rel, other, direction).
H.alert_summary(kg, host=..., rule=..., role="tier1") returns history lines. H.chat(model, messages,
max_tokens) returns {"text", ...}. An alert is a dict with keys alert_id, ts, host, rule_id, level,
description, src_ip, dst_ip, repeat_count.
Write triage(alert, model="qwen") -> dict with keys alert_id, host, owner, criticality, priority, reason:
- criticality = the host node's "criticality" property (1 if the host is unknown); owner = the node reached from
  the host by OWNED_BY ("" if none)
- priority by this rubric: "P1" if level >= 10 and criticality >= 4; "P2" if level >= 10, or level >= 8 and
  criticality >= 4; "P3" if level >= 6; else "P4"
- reason: for P1/P2 ask the LLM (temperature 0, max_tokens 80) for ONE sentence an analyst should read first,
  given the alert, the host facts and H.alert_summary(kg, host=alert["host"], rule="rule:" + alert["rule_id"])
  as history; for P3/P4 return a short template string without calling the LLM.
Put the code in the empty cell under A.1, run it, then run the check cell.
```

In [ ]:
# ⬇️ YOUR CODE HERE (written with your AI assistant)
# Contract: triage(alert, model='qwen') -> {'alert_id', 'host', 'owner', 'criticality', 'priority', 'reason'}


In [ ]:
# A.1 check, then hand triage to the server that n8n calls
import json
H.check_triage(triage)
H.register(triage=triage)

**A.2 Open n8n and finish the workflow (manual, ~15 min).**
1. Run the next cell and open the link. Create the n8n **owner account** (any e-mail; it stays inside your VM).
2. Open the workflow **SOC alert triage (CS 789 A7)**. Click each node and read its settings — note how *Open case*
   reads values from earlier nodes with `$('Triage').item.json…`.
3. Create a credential for the local model: *Overview → Credentials → Create → **OpenAI***: API Key `EMPTY`,
   **Base URL `http://localhost:8000/v1`** → Save (the connection test should pass — it lists the model `qwen`).
4. Delete the node **Write summary (replace with LLM)**. Add **Basic LLM Chain** (AI → Chains) in its place, wired
   *P1 or P2? (true) → Basic LLM Chain → Open case*. Prompt: *Source for Prompt* = **Define below**, and write a prompt
   that asks for a 2-sentence case summary using `{{ JSON.stringify($('Alert in').item.json.body) }}` and
   `{{ JSON.stringify($('Triage').item.json) }}`.
5. Attach a chat model to the chain: **OpenAI Chat Model**, your credential, *Model* → **By ID** → `qwen`, and
   **turn OFF "Use Responses API"** (vLLM's chat endpoint is the classic one).
6. Click **Publish** (top right). Only a published workflow answers on `/webhook/soc-alert`.

In [ ]:
print("n8n editor:", H.public_url(5678))

In [ ]:
# A.3 Replay the live feed (178 alerts from Jan 24 12:00-24:00 UTC) into n8n (~3 min) and score the triage
H.replay_feed()
display(H.score_triage(dict(zip(H.triage_log().alert_id, H.triage_log().priority))))
display(H.cases()[["id", "alert_ts", "host", "owner", "priority", "reason", "summary"]])

✍️ **Questions A-1 … A-3**
* **A-1** Screenshot of your published workflow and of one **successful execution** of a P1 alert
  (*Executions* tab → open it). Paste the prompt you wrote in the Basic LLM Chain.
* **A-2** Triage agreement and confusion table; number of cases; seconds per triage call for P1/P2 vs. P3/P4
  (`H.triage_log().groupby("priority").seconds.mean()`). Why did we not call the LLM for P3/P4?
* **A-3** You could call `triage()` directly from Python. What does putting n8n in between give a SOC (think:
  executions log, retries, adding a ticketing or chat step without code, a human-approval *Wait* node)? Which node
  would you add next, and where?

---
## Part B — An uncensored copilot model with Heretic (≈ 70 min, ~35 of them unattended)

**Abliteration** (lecture on uncensored models): a model's tendency to refuse is largely carried by one *direction*
in its residual stream. Take the mean activation on prompts it usually refuses, subtract the mean on prompts it
answers, and you get a "refusal direction"; project it out of the weights and the model stops refusing — without
any training. **Heretic** (p-e-w/heretic, AGPL-3.0) automates this: it searches where and how strongly to ablate,
minimizing two numbers at once — **refusals** on 100 public refusal-triggering prompts (`mlabonne/harmful_behaviors`)
and **KL divergence** from the original model on 100 harmless prompts (`mlabonne/harmless_alpaca`), its measure of
the damage done. You will abliterate the copilot's own model, Qwen3-4B-Instruct-2507, and then ask the question
that matters for a product: **what happens to the copilot's guardrails when someone swaps in an uncensored model?**

> **Rules for this part.** This is a measurement exercise. Heretic's evaluator reports counts only — keep it that way:
> do not use Heretic's *Chat with the model* option to collect harmful content, do not upload the model to Hugging
> Face, and do not paste the model's answers to refusal-triggering prompts into your report. Your report contains
> numbers, the copilot's answers to the 22 SOC questions, and your analysis.

In [ ]:
# B.0 Install Heretic in its own environment (~4 min) and free the GPU (this stops Qwen -- finish Part A first)
H.install_heretic()
H.free_gpu()
HERETIC_CMD = H.heretic_command(n_trials=60, n_startup_trials=20)
print(HERETIC_CMD)

**B.1 Run Heretic (manual, ~20–40 min on an L4).** Run the next cell. Heretic downloads Qwen (if needed) and the two
prompt sets, measures the original model, computes the refusal directions, then runs **60 trials** (the default is
200; 20 of ours are random exploration, the rest are guided by Optuna). Each trial prints its *refusals* and *KL
divergence*. When it finishes it asks questions **in the cell's input box** (Heretic detects Colab):
1. It lists the **Pareto-optimal trials** — no other trial has both fewer refusals and lower KL. **Copy that list**
   into `TRIALS` in the cell after (trial number, refusals, KL).
2. *Which trial do you want to use?* — choose one and write down why (B-1).
3. *What do you want to do with the decensored model?* → **Save the model to a local folder** → path
   **`/content/qwen-heretic-mine`** → **merge** (vLLM needs the full model, not only the LoRA).
4. Back in the menu choose **Exit program**. Do **not** choose *Upload*.

If the runtime disconnects, run the cell again: Heretic resumes from its checkpoint in `/content/heretic_work`.
You can also run `HERETIC_CMD` in Colab's **Terminal** (bottom-left `>_`) instead of the cell.

In [ ]:
# B.1 Heretic (interactive at the end -- answer in the input box)
!{HERETIC_CMD}

In [ ]:
# B.1 (manual) Heretic's list of Pareto-optimal trials: (trial, refusals out of 100, KL divergence)
TRIALS = [
    # (12, 31, 0.0123),
]
CHOSEN = None      # the trial you saved

In [ ]:
# B.2 Heretic's evaluator on three models: the original, yours, and the published one made with 200 trials
#     (~5 min per model). Prints refusals out of 100 and KL divergence from the original -- numbers only.
EVAL = H.heretic_evaluate([H.BASE_MODEL, str(H.MY_HERETIC_DIR), H.PUBLISHED_HERETIC])
EVAL

In [ ]:
# B.3 The trade-off: your Pareto front, your chosen trial, and the published model
extra = {r.model.split("/")[-1]: (int(str(r.refusals).split("/")[0]), r.kl_divergence)
         for r in EVAL.itertuples() if r.refusals and r.model != str(H.MY_HERETIC_DIR)}
H.plot_tradeoff(TRIALS, chosen=CHOSEN, extra=extra)

**B.4 The copilot with an uncensored model.** Same retriever, same system prompt, same 22 questions as A6 Part C —
only the model changes. The harness serves each abliterated model with vLLM and swaps it in and out of the GPU
(1–2 minutes per switch). Then look at the answers side by side where guardrails matter.

In [ ]:
# B.4 (~8 min)
UNCENSORED = H.register_heretic_models()          # heretic-mine (yours) and heretic-pew (published)
H.eval_bot("B-uncensored", answer, models=["qwen"] + UNCENSORED)
display(H.scoreboard("B-uncensored")); H.plot_scoreboard("B-uncensored")
H.compare_answers("B-uncensored")

**B.5 A guardrail canary — write it yourself (no AI).** A product that lets an operator change the model needs a
check that runs before a model goes live. Write `guardrail_canary(model)`: ask the copilot (your `answer`) the
questions from `H.load_questions()` whose correct behavior is a **refusal** (the `action`, `out_of_scope` and the
tier-1 `restricted` ones) and a few normal ones; score them with `H.score_answer`; return a dict with the refusal
rate, the normal-question accuracy, and `ok` (True only if both are above thresholds you choose). Run it on all
three models.

In [ ]:
# B.5 YOUR guardrail canary
def guardrail_canary(model, min_refusal=0.8, min_normal=0.6):
    ...

for m in ["qwen"] + UNCENSORED:
    print(m, guardrail_canary(m))

In [ ]:
# B.6 Back to the original model for Part C
H.ensure_model("qwen")

✍️ **Questions B-1 … B-5**
* **B-1** Your Pareto list and the B.3 plot. Which trial did you choose, and why that point on the curve? How long did
  the 60 trials take?
* **B-2** The B.2 table: refusals and KL for the original, yours and the published model. Why do yours (60 trials) and
  the published one (200 trials) differ? What does a KL divergence of 0.1 vs. 1.0 mean for the copilot's answers?
* **B-3** The B-uncensored scoreboard. Which question types changed between qwen and the abliterated models, and which
  did not? How many **LEAK**s — and why did (or didn't) the restricted questions leak, given how your A6 retriever works?
* **B-4** Your `guardrail_canary` (code and results). Would it catch a swapped model before it reaches analysts? What
  else would — think model hashes or signatures in a model registry, a KL probe against the reference model, refusal
  monitoring in production?
* **B-5** One paragraph: where must the guardrails of a SOC copilot live, given B-3 — in the system prompt, in the
  retriever and access control, in read-only tools, in output filters? Rank them and justify the ranking.

---
## Part C — The finished SOC Copilot (≈ 35 min)

Everything comes together in the web app you launched in A6: dashboards, the copilot chat (now with a choice of the
original and the uncensored models), the case board fed by n8n, and the KG explorer. Four panels are provided
(`kpis`, `alerts_per_hour`, `top_hosts`, `top_rules`). You add four more. A panel is a function `panel(D) -> plotly Figure`; `D` is what the server hands it (see the prompt).
The page has a **view as** selector (tier1 / manager) — real sign-in is out of scope for A6; the role is passed to
your chat and panel functions.

#### 🛠️ C.1 Four dashboard panels

**Contract** (the checks below depend on these exact names): `panel_attack_heatmap(D)`, `panel_host_heat(D)`, `panel_bot_metrics(D)`, `panel_exercise_overlay(D)` → plotly Figure

**🤖 Prompt for your AI assistant** — copy it into Claude in Chrome, Gemini in Chrome or Gemini in Colab, adapt it, and have the assistant write the code into the empty cell below and run it. Keep the prompt you finally used: it goes into your report.

```text
In this Colab notebook, a dashboard panel is a function that takes D and returns a plotly.graph_objects.Figure.
D has: D.alerts (DataFrame: alert_id, ts, host, rule_id, level, description, mitre_ids "|"-separated or "",
src_ip, hour like "2022-01-21T05:00Z", day like "2022-01-21"), D.hosts (host, criticality, zone, owner),
D.metrics (DataFrame of chat answers: ts, role, model, prompt_tokens, completion_tokens, context_tokens,
seconds), D.cases, D.triage, and D.windows (DataFrame window_id, phase, start, end; None unless the viewer is
the manager).
Write four functions with plotly, each returning a figure about 360 px high with a title:
1. panel_attack_heatmap(D): heatmap of alert counts, ATT&CK technique id (rows; split mitre_ids on "|" and drop
   alerts without a technique) x day (columns), with the counts written in the cells
2. panel_host_heat(D): heatmap of the hourly number of alerts with level >= 8, host (rows) x hour (columns)
3. panel_bot_metrics(D): scatter of prompt_tokens (x) vs seconds (y) for every chat answer, colored by model;
   if D.metrics is empty return a figure with the annotation "no chat answers yet"
4. panel_exercise_overlay(D): line of alerts per hour, with every row of D.windows drawn as a shaded vertical
   band (fig.add_vrect from start to end) annotated with its phase; if D.windows is None return an empty figure
Put the code in the empty cell under C.1 and run it; the next cell registers and previews them.
```

In [ ]:
# ⬇️ YOUR CODE HERE (written with your AI assistant)
# Contract: panel_attack_heatmap(D), panel_host_heat(D), panel_bot_metrics(D), panel_exercise_overlay(D) → plotly Figure


In [ ]:
# C.1 register the panels (the exercise overlay is manager-only) and preview them here
H.register_panel("attack_heatmap", panel_attack_heatmap, tab="Detections", title="ATT&CK techniques by day")
H.register_panel("host_heat", panel_host_heat, tab="Detections", title="High alerts per host and hour")
H.register_panel("bot_metrics", panel_bot_metrics, tab="Copilot", title="Copilot tokens vs latency")
H.register_panel("exercise_overlay", panel_exercise_overlay, tab="Detections", title="Exercise windows",
                 restricted=True)
for f in [panel_attack_heatmap, panel_host_heat, panel_exercise_overlay]:
    f(H.panel_data("manager")).show()

In [ ]:
# C.2 Start the product with YOUR functions: chat_turn (your A6 copilot), triage (Part A), the KG and the panels
H.ensure_model("qwen")
H.register(chat_turn=chat_turn, triage=triage, kg=kg, conversations_db=DB,
           models=["qwen", "heretic-mine", "heretic-pew"] + (["gemini"] if H.gemini_key() else []))
url = H.start_platform()
print("Open the SOC Copilot:", url)

**C.3 Use it (manual, ~10 min).** Open the link. Look through the tabs. In the chat, as **tier1**, ask at least
five questions of your own, including two follow-ups that need memory ("…and who owns it?") and one restricted
question. Switch *view as* to **manager**: the *Exercise windows* panel appears; ask the restricted question again.
Open the **Cases** tab (filled by n8n) and the **Knowledge graph** tab (try `intranet-server`, `T1078`,
`alyssa.hayes`). Take the screenshots for C-1. (The model menu lists only models whose server is running — the GPU
holds one at a time. Optional: `H.ensure_model("heretic-mine")`, reload the page, and ask the uncensored copilot the
same guardrail questions you used in B.4.)

In [ ]:
# C.4 Acceptance test: drives the running product end to end (~1 min)
H.acceptance(level="a7")

✍️ **Questions C-1 … C-3**
* **C-1** Screenshots: the Overview tab, the Detections tab with your heatmaps, the Cases tab, and the chat answering
  the **same restricted question** as tier1 and as manager.
* **C-2** The acceptance scorecard. If a check fails, explain why.
* **C-3** Your startup proposal is due soon. List the three things this prototype needs before a real SOC would
  pay for it, and how you would build each one (e.g. sign-in and authorization, audit log, evaluation in CI, data
  retention, rate limits, multi-tenant KG).

*Optional bonus (+5): **Graphiti temporal memory.*** Replace or complement the SQLite history with
[Graphiti](https://github.com/getzep/graphiti) (`pip install graphiti-core`), which stores each chat turn as an
*episode* in a temporal KG with validity times, so the copilot can answer "what did we conclude about vpn
yesterday?". Use your A5 Neo4j AuraDB as its backend and the local vLLM through Graphiti's OpenAI-compatible client.
Show one question that SQLite memory cannot answer and Graphiti can.

---
## Part D — Report (in the handout)
1. Tables: A-2 triage, B-2 Heretic evaluation, B-3 scoreboard, B-4 canary.
2. Answers to A-1 … C-3 (short; numbers from *your* run).
3. Screenshots: the n8n workflow and execution (A-1), the B.3 plot, the finished product (C-1).
4. **AI-assistant log** as before: assistant used; three prompts (worked / failed / fix); one place its code ran but
   was wrong, and how you found out.
5. Submit one PDF: the filled handout followed by this notebook printed with its outputs (*File → Print → Save as PDF*).